# 04 - Evaluation and Comparison

Compare frozen candidates on CPU. Development games support selection; held-out games follow freezing. This notebook does not run submission compliance.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT)
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Load the Retained Champion

The default final candidate is notebook 03's champion. HPO branches remain separate unless explicitly selected through a documented run.

In [ ]:
league_directory = PROJECT_ROOT / "checkpoints/self_play" / cfg["run_id"]
print(read_json(league_directory / "complete.json"))
league = read_json(league_directory / "league.json")
champion = PROJECT_ROOT / league["champion"]
previous = PROJECT_ROOT / league["history"][-2] if len(league["history"]) > 1 else None
initial = read_json(PROJECT_ROOT / "results" / cfg["run_id"] / "initial_selection.json")
supervised = PROJECT_ROOT / initial["checkpoint"]
print("Candidate:", champion)
print("Hash:", sha256(champion))

## Development Opponents

128 development positions with colours swapped give 256 games per opponent, at 120 seconds plus 0.5 seconds. The preserved harness uses separate agent processes.

In [ ]:
from chess_rl.evaluation import compare_candidates
development_results = compare_candidates(PROJECT_ROOT, champion, cfg, previous=previous)
print(development_results)

## Development Scores

Intervals resample opening pairs or source families. The reference is 50%; draws contribute half a point.

In [ ]:
from chess_rl.plots import plot_matches
display(plot_matches(PROJECT_ROOT, cfg["run_id"], development_results, "development_matches"))

## Controlled Ablations

Compare classical search, policy-only, value-only, hybrid, and supervised weights. Clocks and fixtures are shared.

In [ ]:
RUN_ABLATIONS = True
if RUN_ABLATIONS:
    from chess_rl.evaluation import run_ablations
    ablation_results = run_ablations(PROJECT_ROOT, champion, cfg, supervised)
    display(plot_matches(PROJECT_ROOT, cfg["run_id"], ablation_results, "ablations"))

## Freeze and Evaluate Held-Out Games

Complete all selected training and tuning first. This locks the checkpoint and search config. A different model cannot reuse this suite as if it had been preselected.

In [ ]:
heldout_results = compare_candidates(PROJECT_ROOT, champion, cfg, previous=previous, heldout=True)
display(plot_matches(PROJECT_ROOT, cfg["run_id"], heldout_results, "heldout_matches"))

## Record Final Selection

This manifest unlocks notebook 05, recording actual completed training and evaluation.

In [ ]:
from chess_rl.evaluation import finalize_selection
selection_path = finalize_selection(PROJECT_ROOT, champion, cfg, heldout_results)
print("Frozen selection:", selection_path)
print("Notebook 04 complete. Open notebook 05.")